# Modul 5: Convolutional Neural Network Dasar

**Nama:** ISI NAMA
**NIM:** ISI NIM
**Kelas:** ISI KELAS
**Tanggal:** YYYY-MM-DD

Simpan berkas ini sebagai `M05_NIM.ipynb` sebelum mulai mengerjakan.

## Petunjuk

1. Ganti seluruh penanda `TODO`. Jangan menghapus sel pemeriksaan yang berisi `cek(...)` atau `assert`.
2. Setiap isian **hitungan tangan** diisi sebelum sel kode sesudahnya dijalankan.
3. Implementasi manual hanya boleh memakai indexing, perkalian elemen, penjumlahan, `F.pad`, dan `torch.flip`. `F.conv2d` dan autograd hanya dipakai sebagai pembanding.
4. Seluruh verifikasi memakai `torch.float64` dan `torch.allclose`.
5. Protokol tugas: subset $10\,000$/$2\,000$, Adam $10^{-3}$, batch $128$, $10$ epoch ($790$ update). Bila memakai Fashion-MNIST karena keterbatasan CPU, sebutkan pada laporan.
6. Luaran: `M05_NIM.ipynb`, `M05_NIM.pdf`, dan `M05_NIM_metrics.csv`.

In [ ]:
import platform
import random
import statistics
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import Markdown, display
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import CIFAR10, FashionMNIST

DATASET = 'cifar10'      # ganti ke 'fashion' bila hanya tersedia CPU
NIM = 'TODO'                 # contoh: '120450123'
SEED = int(str(NIM)[-4:]) if str(NIM).isdigit() else 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DT = torch.float64

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def cek(nama, hasil, acuan):
    """Sel pemeriksaan: shape harus sama dan nilai lolos torch.allclose."""
    assert hasil is not None and acuan is not None, f'{nama}: TODO belum diisi'
    hasil = torch.as_tensor(hasil.detach() if torch.is_tensor(hasil) else hasil, dtype=DT)
    acuan = torch.as_tensor(acuan.detach() if torch.is_tensor(acuan) else acuan, dtype=DT)
    ok = hasil.shape == acuan.shape and torch.allclose(hasil, acuan)
    print(f'[{"OK" if ok else "GAGAL"}] {nama}: shape {tuple(hasil.shape)}')
    assert ok, f'{nama} tidak cocok dengan acuan'

def ukur(fungsi, ulang=10):
    """Median waktu eksekusi `fungsi` dalam detik setelah satu pemanasan."""
    fungsi()
    waktu = []
    for _ in range(ulang):
        mulai = time.perf_counter()
        fungsi()
        waktu.append(time.perf_counter() - mulai)
    return statistics.median(waktu)

seed_everything(SEED)
pd.set_option('display.precision', 4)
print({'python': platform.python_version(), 'torch': torch.__version__,
       'device': str(DEVICE), 'dataset': DATASET, 'seed': SEED})

## Pre-lab — 10 poin

Kerjakan **sebelum** sesi tanpa menjalankan kode. Gunakan $\mathbf{X}$ dan $\mathbf{K}$ pada Bagian A.

a. **$\mathbf{X}\star\mathbf{K}$ dengan $p=1$ dan $s=2$ (tulis shape, langkah $Y_{0,0}$, dan seluruh elemen):** TODO

b. **Jendela yang menjadi kolom ke-4 pada `F.unfold(X, 3)`:** TODO

c. **Parameter dan jumlah perkalian `Conv2d(3, 32, kernel_size=3, padding=1)` pada masukan $3\times32\times32$:** TODO

d. **Mengapa penamaan convolution untuk cross-correlation tidak mengubah hasil pelatihan:** TODO

## A. Cross-correlation manual — bagian dari 20 poin

$$\mathbf{X}=\begin{bmatrix}1&2&0&1&3\\0&1&3&2&1\\2&1&0&1&2\\1&0&2&3&0\\3&1&1&0&2\end{bmatrix},\qquad
\mathbf{K}=\begin{bmatrix}1&0&-1\\2&1&0\\0&-1&1\end{bmatrix}$$

**Hitungan tangan $\mathbf{X}\star\mathbf{K}$ ($p=0$, $s=1$).** Tuliskan langkah $Y_{0,0}$ dan $Y_{0,1}$, lalu seluruh matriks: TODO

In [ ]:
X = torch.tensor([[1, 2, 0, 1, 3],
                  [0, 1, 3, 2, 1],
                  [2, 1, 0, 1, 2],
                  [1, 0, 2, 3, 0],
                  [3, 1, 1, 0, 2]], dtype=DT)
K = torch.tensor([[1, 0, -1],
                  [2, 1, 0],
                  [0, -1, 1]], dtype=DT)

def corr2d(X, K):
    """TODO A1: cross-correlation 2D satu kanal, tanpa padding, stride 1.

    Hanya indexing, perkalian elemen, dan penjumlahan.
    """
    raise NotImplementedError

Y_TANGAN = None  # TODO A2: salin hasil hitungan tangan, contoh [[a, b, c], [d, e, f], [g, h, i]]

Y_manual = corr2d(X, K)
print('X ⋆ K =\n', Y_manual)
cek('hitungan tangan vs corr2d', Y_manual, Y_TANGAN)
cek('corr2d vs F.conv2d', Y_manual, F.conv2d(X[None, None], K[None, None])[0, 0])

Y_konvolusi = corr2d(X, torch.flip(K, (0, 1)))
print('X * K (kernel dibalik) =\n', Y_konvolusi)
assert not torch.allclose(Y_konvolusi, Y_manual), 'konvolusi matematis seharusnya berbeda'

**Konvolusi matematis.** Mengapa $\mathbf{X}*\mathbf{K}$ berbeda seluruhnya dari $\mathbf{X}\star\mathbf{K}$, bukan hanya berbeda tanda? Kapan keduanya hanya berbeda tanda? TODO

## B. Padding, stride, dan banyak kanal — bagian dari 20 poin

$$Y_{c',i,j}=b_{c'}+\sum_{c}\sum_{u=0}^{k-1}\sum_{v=0}^{k-1}W_{c',c,u,v}\,\tilde{X}_{c,\,si+u,\,sj+v},\qquad
n_\text{keluar}=\left\lfloor\frac{n+2p-d(k-1)-1}{s}\right\rfloor+1$$

In [ ]:
def shape_keluar(n, k, p=0, s=1, d=1):
    """TODO B1: rumus shape satu sumbu spasial, termasuk dilation."""
    raise NotImplementedError

def corr2d_multi(X, W, b=None, stride=1, padding=0):
    """TODO B2: persamaan banyak kanal pada modul.

    X: (C_in, H, W), W: (C_out, C_in, k, k), b: (C_out,) atau None.
    Kembalikan (C_out, H_out, W_out). Padding dibuat dengan F.pad; operasi
    utamanya hanya indexing, perkalian elemen, dan penjumlahan.
    """
    raise NotImplementedError

PRELAB_A = None  # TODO B3: salin jawaban pre-lab butir a

Y_s2 = corr2d_multi(X[None], K[None, None], stride=2, padding=1)[0]
print('p=1, s=2:\n', Y_s2)
cek('pre-lab a vs corr2d_multi', Y_s2, PRELAB_A)
cek('corr2d_multi vs F.conv2d (p=1, s=2)', Y_s2,
    F.conv2d(X[None, None], K[None, None], stride=2, padding=1)[0, 0])

Xm = torch.tensor([[[1, 0, 2], [3, 1, 0], [0, 2, 1]],
                   [[2, 1, 0], [0, 1, 3], [1, 0, 2]]], dtype=DT)
Wm = torch.tensor([[[[1, -1], [0, 2]], [[0, 1], [-1, 1]]],
                   [[[1, 0], [0, 1]], [[1, 1], [1, 1]]]], dtype=DT)
bm = torch.tensor([0.5, -1.0], dtype=DT)
Ym_modul = torch.tensor([[[5.5, 0.5], [6.5, 8.5]],
                         [[5.0, 4.0], [6.0, 7.0]]], dtype=DT)

Ym = corr2d_multi(Xm, Wm, bm)
cek('contoh dua kanal vs modul', Ym, Ym_modul)

conv = nn.Conv2d(2, 2, 2).double()

# TODO B4: salin Wm dan bm ke conv.weight dan conv.bias di dalam torch.no_grad().
...

cek('contoh dua kanal vs nn.Conv2d', conv(Xm[None])[0], Ym)
print('parameter nn.Conv2d(2, 2, 2):', sum(q.numel() for q in conv.parameters()))

In [ ]:
KONFIGURASI = [  # (C_in, C_out, n, k, s, p)
    (3, 4, 7, 3, 1, 1),
    (3, 4, 9, 5, 2, 2),
    (1, 2, 6, 3, 2, 0),
    (2, 3, 8, 5, 1, 0),
    (3, 4, 11, 3, 2, 1),
]
gen = torch.Generator().manual_seed(SEED)
baris = []
for c_in, c_out, n, k, s, p in KONFIGURASI:
    Xr = torch.randn(c_in, n, n, dtype=DT, generator=gen)
    Wr = torch.randn(c_out, c_in, k, k, dtype=DT, generator=gen)
    br = torch.randn(c_out, dtype=DT, generator=gen)
    Y_manual = corr2d_multi(Xr, Wr, br, stride=s, padding=p)
    Y_acuan = F.conv2d(Xr[None], Wr, br, stride=s, padding=p)[0]
    shape_rumus = (c_out, shape_keluar(n, k, p, s), shape_keluar(n, k, p, s))
    baris.append({'C_in': c_in, 'C_out': c_out, 'n': n, 'k': k, 's': s, 'p': p,
                  'shape_rumus': shape_rumus, 'shape_manual': tuple(Y_manual.shape),
                  'shape_cocok': shape_rumus == tuple(Y_acuan.shape) == tuple(Y_manual.shape),
                  'allclose': torch.allclose(Y_manual, Y_acuan)})
tabel_b = pd.DataFrame(baris)
display(tabel_b)
assert tabel_b['shape_cocok'].all() and tabel_b['allclose'].all(), 'ada konfigurasi yang belum lolos'
print('kelima konfigurasi lolos')

**Pembulatan ke bawah.** Pada konfigurasi `(1, 2, 6, 3, 2, 0)`, baris dan kolom masukan mana yang tidak pernah dikunjungi jendela? TODO

## C. Unfold dan biaya komputasi — 10 poin

$$\operatorname{vec}(\mathbf{Y})=\mathbf{W}_\text{flat}\,\mathbf{U}+\mathbf{b}\mathbf{1}^\top,\qquad
\mathbf{U}\in\mathbb{R}^{C_\text{in}k^2\times L},\qquad
\text{perkalian}=C_\text{out}C_\text{in}k^2H_\text{keluar}W_\text{keluar}$$

In [ ]:
PRELAB_B = None  # TODO C1: salin kolom ke-4 dari pre-lab butir b

U = None         # TODO C2: F.unfold pada X berukuran (1, 1, 5, 5) dengan kernel 3
Y_unfold = None  # TODO C3: K.reshape(1, -1) @ U, lalu ubah kembali menjadi (3, 3)

print('shape U:', tuple(U.shape))
assert tuple(U.shape) == (1, 9, 9)
cek('kolom ke-4 U vs pre-lab b', U[0, :, 4], PRELAB_B)
cek('unfold vs corr2d', Y_unfold, corr2d(X, K))

def conv_unfold(X, W, b=None, stride=1, padding=0):
    """TODO C4: X (C_in, H, W) -> (C_out, H_out, W_out) memakai F.unfold dan @."""
    raise NotImplementedError

c_in, c_out, n, k, s, p = KONFIGURASI[1]
Xr = torch.randn(c_in, n, n, dtype=DT, generator=gen)
Wr = torch.randn(c_out, c_in, k, k, dtype=DT, generator=gen)
br = torch.randn(c_out, dtype=DT, generator=gen)
cek('conv_unfold vs F.conv2d', conv_unfold(Xr, Wr, br, s, p),
    F.conv2d(Xr[None], Wr, br, stride=s, padding=p)[0])

In [ ]:
def matriks_konvolusi(K, n):
    """TODO C5: bangun M sehingga vec(corr2d(X, K)) = M @ vec(X) untuk X berukuran n x n.

    Petunjuk: kolom ke-q dari M adalah vec(corr2d(e_q, K)), dengan e_q citra satu-hot.
    """
    raise NotImplementedError

M = matriks_konvolusi(K, 5)
print('shape M:', tuple(M.shape), '| entri tak nol:', int((M != 0).sum()), 'dari', M.numel())
cek('M @ vec(X) vs vec(X ⋆ K)', M @ X.flatten(), corr2d(X, K).flatten())
assert all(torch.equal(M[r][M[r] != 0], K[K != 0]) for r in range(M.shape[0])), \
    'setiap baris M seharusnya memakai bobot yang sama (parameter sharing)'
print('setiap baris M memakai bilangan yang sama dengan K')

In [ ]:
torch.manual_seed(SEED)
Xb = torch.randn(8, 3, 32, 32)        # float32 cukup untuk pengukuran waktu
Wb = torch.randn(16, 3, 3, 3)
bb = torch.randn(16)

# TODO C6: isi tiga fungsi tanpa argumen yang memproses SELURUH batch Xb dengan padding=1.
implementasi = {
    'corr2d_multi (loop)': None,
    'conv_unfold (matmul)': None,
    'F.conv2d': None,
}

rujukan = F.conv2d(Xb, Wb, bb, padding=1)
baris = []
for nama, fungsi in implementasi.items():
    assert fungsi is not None, f'{nama}: TODO belum diisi'
    assert torch.allclose(fungsi(), rujukan, atol=1e-4), f'{nama} tidak cocok dengan F.conv2d'
    baris.append({'implementasi': nama, 'median_detik': ukur(fungsi)})
tabel_waktu = pd.DataFrame(baris)
tabel_waktu['kali_lebih_lambat'] = tabel_waktu['median_detik'] / tabel_waktu['median_detik'].iloc[-1]
display(tabel_waktu)

**Entri tak nol dan waktu.** Berapa entri tak nol pada $\mathbf{M}$ dan mengapa kurang dari $9\times9=81$? Urutkan ketiga implementasi dari yang tercepat dan jelaskan sumber selisihnya. TODO

**Checkpoint menit ke-70.** Tunjukkan kepada asisten: hitungan tangan Bagian A dan pre-lab yang lolos `cek`, tabel lima konfigurasi, dan tabel waktu.

## D. Gradien manual lawan autograd — bagian dari 20 poin

$$\mathbf{X}=\begin{bmatrix}1&2&0\\0&1&3\\2&1&1\end{bmatrix},\quad
\mathbf{K}=\begin{bmatrix}1&-1\\2&0\end{bmatrix},\quad
\mathbf{G}=\frac{\partial\mathcal{L}}{\partial\mathbf{Y}}=\begin{bmatrix}1&0\\-1&2\end{bmatrix},\qquad
\frac{\partial\mathcal{L}}{\partial\mathbf{K}}=\mathbf{X}\star\mathbf{G},\;\;
\frac{\partial\mathcal{L}}{\partial b}=\sum G_{i,j},\;\;
\frac{\partial\mathcal{L}}{\partial\mathbf{X}}=\operatorname{pad}_{k-1}(\mathbf{G})\star\operatorname{rot180}(\mathbf{K})$$

**Hitungan tangan** untuk $\mathcal{L}=\sum_{i,j}G_{i,j}Y_{i,j}$ dan $b=0$:

- $\mathbf{Y}$: TODO
- $\partial\mathcal{L}/\partial\mathbf{K}$: TODO
- $\partial\mathcal{L}/\partial b$: TODO
- $\partial\mathcal{L}/\partial\mathbf{X}$, termasuk langkah untuk $X_{1,1}$: TODO

In [ ]:
Xg = torch.tensor([[1, 2, 0], [0, 1, 3], [2, 1, 1]], dtype=DT)
Kg = torch.tensor([[1, -1], [2, 0]], dtype=DT)
G = torch.tensor([[1, 0], [-1, 2]], dtype=DT)

DK_TANGAN = None  # TODO D1: salin hasil hitungan tangan
DB_TANGAN = None  # TODO D1: dalam bentuk [nilai]
DX_TANGAN = None  # TODO D1

def gradien_manual(X, K, G):
    """TODO D2: persamaan gradien modul untuk stride 1 tanpa padding.

    Kembalikan (dK, db, dX) dengan db berbentuk (1,). Gunakan corr2d, F.pad, dan torch.flip.
    """
    raise NotImplementedError

dK, db, dX = gradien_manual(Xg, Kg, G)

X_ag = Xg.clone().requires_grad_(True)
K_ag = Kg.clone().requires_grad_(True)
b_ag = torch.zeros(1, dtype=DT, requires_grad=True)

# TODO D3: hitung Y_ag = F.conv2d(...) dari X_ag, K_ag, b_ag, lalu backward dari (G * Y_ag).sum().
...

for nama, manual, tangan, acuan in [('dL/dK', dK, DK_TANGAN, K_ag.grad),
                                    ('dL/db', db, DB_TANGAN, b_ag.grad),
                                    ('dL/dX', dX, DX_TANGAN, X_ag.grad)]:
    cek(f'{nama} tangan vs manual', manual, tangan)
    cek(f'{nama} manual vs autograd', manual, acuan)

In [ ]:
P = torch.tensor([[1, 3, 2, 0], [4, 2, 1, 5], [0, 1, 3, 2], [2, 6, 0, 1]], dtype=DT)
G_pool = torch.tensor([[1, 2], [3, 4]], dtype=DT)

def maxpool_manual(P, G_pool, k=2):
    """TODO D4: kembalikan (keluaran, dL/dP) untuk max pooling k x k dengan stride k."""
    raise NotImplementedError

Z, dP = maxpool_manual(P, G_pool)
P_ag = P.clone().requires_grad_(True)
Z_ag = F.max_pool2d(P_ag[None, None], 2)[0, 0]
(G_pool * Z_ag).sum().backward()
cek('max pooling manual vs F.max_pool2d', Z, Z_ag)
cek('gradien max pooling vs autograd', dP, P_ag.grad)
print('dL/dP =\n', dP)

**Pembalikan kernel dan pooling.** Mengapa gradien terhadap masukan memakai kernel yang dibalik, sedangkan gradien terhadap kernel tidak? Mengapa hanya empat posisi $\mathbf{P}$ yang menerima gradien? TODO

## E. Dari operasi ke jaringan — 10 poin

Isi tabel berikut **sebelum** membangun model. Masukan CIFAR-10 $3\times32\times32$.

| Layer | Keluaran | Parameter | Perkalian | $r$ |
|---|---|---|---|---|
| `Conv2d(3,32,k=3,p=1)` | TODO | TODO | TODO | TODO |
| `MaxPool2d(2)` | TODO | 0 | 0 | TODO |
| `Conv2d(32,64,k=3,p=1)` | TODO | TODO | TODO | TODO |
| `MaxPool2d(2)` | TODO | 0 | 0 | TODO |
| `Linear(?,128)` | TODO | TODO | TODO | — |
| `Linear(128,10)` | TODO | TODO | TODO | — |
| **Total CNN** | | TODO | TODO | |
| **FNN** $3072\rightarrow177\rightarrow10$ | | TODO | TODO | — |

In [ ]:
def buat_cnn(c1=32, c2=64, k=3, c_in=3, hw=32):
    """TODO E1: Conv-ReLU-MaxPool dua kali, Flatten, Linear(c2*(hw//4)**2, 128), ReLU, Linear(128, 10).

    Padding p = k // 2. Panggil seed_everything(SEED) lebih dahulu.
    """
    raise NotImplementedError

def buat_fnn(hidden=177, c_in=3, hw=32):
    """TODO E2: Flatten -> Linear(c_in*hw*hw, hidden) -> ReLU -> Linear(hidden, 10)."""
    raise NotImplementedError

def anatomi(model, shape_masukan):
    """TODO E3: tabel per child module dengan kolom layer, modul, keluaran, parameter, perkalian, rf.

    - pasang forward hook pada setiap child module, lalu jalankan satu citra shape_masukan;
    - perkalian Conv2d = C_out * C_in * k_h * k_w * H_out * W_out; Linear = in * out; lainnya 0;
    - rf: r = r + (k - 1) * j dan j = j * s untuk Conv2d dan MaxPool2d; None setelah Flatten.
    """
    raise NotImplementedError

tabel_cnn = anatomi(buat_cnn(), (3, 32, 32))
tabel_fnn = anatomi(buat_fnn(), (3, 32, 32))
display(tabel_cnn)
display(tabel_fnn)

ringkas = pd.DataFrame({
    'model': ['CNN baseline', 'FNN pembanding'],
    'parameter': [int(tabel_cnn['parameter'].sum()), int(tabel_fnn['parameter'].sum())],
    'perkalian': [int(tabel_cnn['perkalian'].sum()), int(tabel_fnn['perkalian'].sum())],
})
display(ringkas)
assert ringkas['parameter'].tolist() == [545_098, 545_701], 'jumlah parameter belum sesuai modul'
assert ringkas['perkalian'].tolist() == [6_128_896, 545_514], 'jumlah perkalian belum sesuai modul'
assert int(tabel_cnn['rf'].dropna().iloc[-1]) == 10, 'receptive field keluaran blok kedua harus 10'

porsi_fc = tabel_cnn.loc[tabel_cnn['modul'] == 'Linear', 'parameter'].iloc[0] / ringkas.loc[0, 'parameter']
porsi_conv = tabel_cnn.loc[tabel_cnn['modul'] == 'Conv2d', 'perkalian'].sum() / ringkas.loc[0, 'perkalian']
rasio_perkalian = ringkas.loc[0, 'perkalian'] / ringkas.loc[1, 'perkalian']
print(f'parameter pada Linear pertama: {100 * porsi_fc:.1f}% | perkalian pada Conv2d: {100 * porsi_conv:.1f}%')
print(f'rasio perkalian CNN/FNN: {rasio_perkalian:.2f}')

In [ ]:
def rf_empiris(pool):
    """TODO E4: receptive field empiris.

    Susun Conv2d(3,32,3,p=1) -> pool(2) -> Conv2d(32,64,3,p=1) -> pool(2) TANPA ReLU (float64),
    isi bobot convolution dengan 1 dan bias dengan 0, hitung gradien unit keluaran [0, 0, 4, 4]
    terhadap citra acak positif (1, 3, 32, 32), lalu kembalikan (tinggi, lebar) kotak terkecil
    yang memuat seluruh piksel bergradien tak nol.
    """
    raise NotImplementedError

rf_avg, rf_max = rf_empiris(nn.AvgPool2d), rf_empiris(nn.MaxPool2d)
print('AvgPool2d:', rf_avg, '| MaxPool2d:', rf_max)
assert rf_avg == (10, 10), 'receptive field empiris dengan AvgPool2d harus 10 x 10'

In [ ]:
# TODO E5: ukur median waktu forward kedua model (model.eval() dan torch.no_grad())
#          pada batch acak (128, 3, 32, 32) di CPU memakai fungsi ukur.
t_cnn, t_fnn = None, None

print(f'forward CNN: {1e3 * t_cnn:.2f} ms | FNN: {1e3 * t_fnn:.2f} ms')
print(f'rasio waktu CNN/FNN: {t_cnn / t_fnn:.2f} | rasio perkalian: {rasio_perkalian:.2f}')

**Receptive field dan waktu.** Mengapa wilayah bergradien mengecil ketika memakai `MaxPool2d`? Apakah rasio waktu forward mendekati rasio perkalian, dan faktor apa yang diduga membuat keduanya berbeda? TODO

### Latihan mandiri di kelas

Digit terakhir NIM 0–4: kernel blok pertama menjadi $5\times5$ dengan $p=2$. Digit 5–9: kanal keluaran blok pertama menjadi $64$.

**Prediksi tangan** perubahan parameter, perkalian per citra, dan receptive field keluaran blok kedua, beserta layer lain yang ikut berubah: TODO

In [ ]:
digit = int(str(NIM)[-1]) if str(NIM).isdigit() else 0
print('digit terakhir NIM:', digit)

PREDIKSI = {'parameter': None, 'perkalian': None, 'rf': None}  # TODO: salin prediksi tangan

# TODO: bangun varian dari buat_cnn() dengan mengganti modul blok pertama sesuai digit NIM.
#       Sesuaikan layer lain seperlunya agar shape tetap valid.
varian = None

tabel_varian = anatomi(varian, (3, 32, 32))
display(tabel_varian)
aktual = {'parameter': int(tabel_varian['parameter'].sum()),
          'perkalian': int(tabel_varian['perkalian'].sum()),
          'rf': int(tabel_varian['rf'].dropna().iloc[-1])}
print('prediksi:', PREDIKSI)
print('aktual  :', aktual)

## Tugas T1. Gradien pada stride 2 — bagian dari 20 poin Bagian D

Sisipkan $s-1$ nol di antara elemen $\mathbf{G}$, lalu pakai persamaan gradien stride 1. Bila $(n+2p-k)\bmod s\neq0$, potong sisi akhir masukan berpadding untuk $\partial\mathcal{L}/\partial\mathbf{W}$ dan tambahkan padding yang sama di sisi akhir $\mathbf{G}$ untuk $\partial\mathcal{L}/\partial\mathbf{X}$.

In [ ]:
def gradien_conv(X, W, G, stride=1, padding=0):
    """TODO T1: gradien Conv2d banyak kanal tanpa F.conv2d maupun autograd.

    X: (C_in, n, n), W: (C_out, C_in, k, k), G: (C_out, H_out, W_out). Kembalikan (dW, dX).
    Gunakan corr2d/corr2d_multi, F.pad, dan torch.flip.
    """
    raise NotImplementedError

KASUS_T1 = [(1, 1, 5, 3, 2, 0), (2, 3, 6, 3, 2, 1), (3, 2, 7, 3, 2, 0), (2, 2, 8, 5, 2, 1)]  # (C_in, C_out, n, k, s, p)
gen = torch.Generator().manual_seed(SEED)
baris = []
for c_in, c_out, n, k, s, p in KASUS_T1:
    Xr = torch.randn(c_in, n, n, dtype=DT, generator=gen).requires_grad_(True)
    Wr = torch.randn(c_out, c_in, k, k, dtype=DT, generator=gen).requires_grad_(True)
    Y = F.conv2d(Xr[None], Wr, stride=s, padding=p)[0]
    Gr = torch.randn(Y.shape, dtype=DT, generator=gen)
    (Gr * Y).sum().backward()
    dW, dX = gradien_conv(Xr.detach(), Wr.detach(), Gr, s, p)
    dX_api = torch.nn.grad.conv2d_input((1, c_in, n, n), Wr.detach(), Gr[None], stride=s, padding=p)[0]
    baris.append({'kasus (C_in,C_out,n,k,s,p)': (c_in, c_out, n, k, s, p),
                  '(n+2p-k) mod s': (n + 2 * p - k) % s,
                  'dW vs autograd': torch.allclose(dW, Wr.grad),
                  'dX vs autograd': torch.allclose(dX, Xr.grad),
                  'dX vs conv2d_input': torch.allclose(dX, dX_api)})
tabel_t1 = pd.DataFrame(baris)
display(tabel_t1)
assert (tabel_t1['(n+2p-k) mod s'] != 0).any(), 'harus ada kasus dengan sisa tidak nol'
assert tabel_t1.iloc[:, 2:].all().all(), 'ada gradien yang belum cocok'
print('keempat kasus lolos')

**Penjelasan T1.** Mengapa $\mathbf{G}$ perlu disisipi nol, dan apa yang terjadi pada gradien baris atau kolom yang tidak dikunjungi jendela? TODO

## Tugas T2. FNN lawan CNN pada anggaran setara — bagian dari 20 poin

In [ ]:
DATA_ROOT = Path('../../data/raw')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('data/raw')

if DATASET == 'cifar10':
    tr = CIFAR10(root=DATA_ROOT, train=True, download=False)
    te = CIFAR10(root=DATA_ROOT, train=False, download=False)
    X_penuh = torch.tensor(tr.data).permute(0, 3, 1, 2).float() / 255.0
    y_penuh = torch.tensor(tr.targets)
    X_uji_mentah = torch.tensor(te.data).permute(0, 3, 1, 2).float() / 255.0
    y_uji = torch.tensor(te.targets)
else:
    tr = FashionMNIST(root=DATA_ROOT, train=True, download=False)
    te = FashionMNIST(root=DATA_ROOT, train=False, download=False)
    X_penuh = tr.data.unsqueeze(1).float() / 255.0
    y_penuh = tr.targets
    X_uji_mentah = te.data.unsqueeze(1).float() / 255.0
    y_uji = te.targets
KELAS = tr.classes
C_IN, H, W_IMG = X_penuh.shape[1:]

# TODO T2a: ambil 10.000 latih dan 2.000 validasi terstratifikasi dari data latih resmi (random_state=SEED).
idx_latih, idx_val = ..., ...

# TODO T2b: MEAN dan STD per kanal dari subset latih saja (dim=(0, 2, 3), keepdim=True).
MEAN, STD = ..., ...

X_latih, y_latih = X_penuh[idx_latih], y_penuh[idx_latih]
X_val, y_val = X_penuh[idx_val], y_penuh[idx_val]
normalkan = lambda t: (t - MEAN) / STD
ds_latih = TensorDataset(normalkan(X_latih), y_latih)
ds_val = TensorDataset(normalkan(X_val), y_val)

print(f'citra {C_IN}x{H}x{W_IMG} | latih {len(ds_latih)} | validasi {len(ds_val)}')
print('mean per kanal:', MEAN.flatten().tolist(), '| std per kanal:', STD.flatten().tolist())
assert (len(ds_latih), len(ds_val)) == (10_000, 2_000)
assert torch.bincount(y_latih).min().item() == 1_000, 'subset harus terstratifikasi'
print('split sesuai protokol')

In [ ]:
BATCH, EPOCH = 128, 10

cnn_tugas = buat_cnn(c_in=C_IN, hw=H)
p_cnn = sum(q.numel() for q in cnn_tugas.parameters())
d_in = C_IN * H * W_IMG
kandidat = range(max(1, (p_cnn - 10) // (d_in + 11) - 2), (p_cnn - 10) // (d_in + 11) + 3)
hidden = min(kandidat, key=lambda h: abs((d_in + 11) * h + 10 - p_cnn))   # P(h) = (d+1)h + 10(h+1)
fnn_tugas = buat_fnn(hidden, c_in=C_IN, hw=H)
p_fnn = sum(q.numel() for q in fnn_tugas.parameters())
print(f'hidden={hidden} | parameter FNN={p_fnn:,} | CNN={p_cnn:,} | selisih={100 * abs(p_fnn - p_cnn) / p_cnn:.3f}%')
assert abs(p_fnn - p_cnn) / p_cnn < 0.01, 'anggaran belum setara (selisih > 1%)'

def loader(ds, batch, acak):
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(ds, batch_size=batch, shuffle=acak, generator=g if acak else None)

@torch.no_grad()
def evaluasi(model, ds, batch=512):
    """TODO T2c: kembalikan (loss rata-rata, akurasi). Jangan lupa model.eval()."""
    raise NotImplementedError

def jalankan(model, label, epoch=EPOCH):
    """TODO T2d: satu fungsi pelatihan untuk seluruh run.

    Adam lr=1e-3 dan CrossEntropyLoss. Catat val_loss dan val_acc setiap epoch.
    Kembalikan (riwayat, catatan); catatan memuat run_id, seed, dataset, arsitektur,
    parameter, perkalian (dari anatomi), n_update, train_loss, val_loss, val_acc,
    gap = val_loss - train_loss, detik_per_epoch, dan catatan.
    """
    raise NotImplementedError

hasil, kurva, model_terlatih = [], {}, {}
for model, label in [(fnn_tugas, 'fnn'), (cnn_tugas, 'cnn-baseline')]:
    riwayat, catatan = jalankan(model, label)
    hasil.append(catatan)
    kurva[label] = riwayat
    model_terlatih[label] = model

tabel = pd.DataFrame(hasil)
display(tabel[['run_id', 'parameter', 'perkalian', 'n_update', 'train_loss', 'val_loss',
               'val_acc', 'gap', 'detik_per_epoch']])
assert set(tabel['n_update']) == {790}, 'kedua model harus memakai 790 update'

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
for label, r in kurva.items():
    axes[0].plot(r['epoch'], r['val_loss'], marker='o', label=label)
    axes[1].plot(r['epoch'], r['val_acc'], marker='o', label=label)
axes[0].set(xlabel='epoch', ylabel='validation loss', title='Validation loss')
axes[1].set(xlabel='epoch', ylabel='validation accuracy', title='Validation accuracy')
for ax in axes:
    ax.grid(alpha=.3)
    ax.legend()
fig.suptitle(f'FNN dan CNN pada anggaran parameter setara ({DATASET})')
plt.tight_layout()
plt.show()

**Penjelasan T2.** Model mana yang menang pada anggaran parameter setara dan berapa selisihnya? Bandingkan rasio waktu per epoch dengan rasio perkalian. TODO

## Tugas T3. Kernel, feature map, dan kernel Sobel manual — bagian dari 20 poin

In [ ]:
# TODO T3a: tampilkan 32 kernel Conv2d pertama CNN baseline setelah pelatihan, beri indeks kanal.
# TODO T3b: tampilkan delapan feature map setelah ReLU dari blok 1 dan blok 2 untuk ds_val[0],
#           dengan keterangan blok dan indeks kanal.
# TODO T3c: terapkan kernel Sobel x dan y memakai corr2d pada citra yang sama (rata-rata kanal,
#           float64), verifikasi dengan cek terhadap F.conv2d, lalu tampilkan hasil dan magnitudonya.
raise NotImplementedError

**Penjelasan T3.** Apa beda watak feature map blok pertama dan blok kedua? Kernel mana yang tampak menyerupai detektor tepi Sobel? TODO

## Tugas T4. Evaluasi akhir — bagian dari 20 poin

In [ ]:
# TODO T4a: pilih model dengan validation loss terendah.
# TODO T4b: confusion matrix pada validation set dan pasangan kelas yang paling sering tertukar.
# TODO T4c: tampilkan LIMA prediksi salah beserta label benar dan label prediksi.
# TODO T4d: baru pada tahap ini buat ds_uji dan evaluasi test SATU KALI; catat angkanya.
raise NotImplementedError

**Analisis lima kesalahan.**

| No | Label benar | Prediksi | Dugaan penyebab |
|----|-------------|----------|-----------------|
| 1 | TODO | TODO | TODO |
| 2 | TODO | TODO | TODO |
| 3 | TODO | TODO | TODO |
| 4 | TODO | TODO | TODO |
| 5 | TODO | TODO | TODO |

In [ ]:
tabel_metrics = tabel.copy()
tabel_metrics.insert(0, 'module', 'M05')
tabel_metrics.insert(1, 'student_id', NIM)
tabel_metrics.to_csv(f'M05_{NIM}_metrics.csv', index=False)
print(f'{len(tabel_metrics)} baris tersimpan ke M05_{NIM}_metrics.csv')
assert len(tabel_metrics) == 2, 'metrics.csv harus memuat run FNN dan CNN baseline'

## Pertanyaan analisis — 10 poin

1. Mengapa gradien terhadap masukan memerlukan kernel yang dibalik, sedangkan gradien terhadap kernel tidak? Jelaskan dengan hitungan $\partial\mathcal{L}/\partial X_{1,1}$ dari Bagian D. TODO
2. Pada stride $2$, apa yang terjadi pada baris dan kolom masukan yang tidak pernah dikunjungi jendela, dan bagaimana gradiennya? TODO
3. Pada anggaran parameter yang praktis sama, model mana yang menang dan berapa selisih accuracy-nya? Kaitkan dengan sparse connectivity dan parameter sharing. TODO
4. Rasio perkalian CNN terhadap FNN sekitar sebelas kali. Berapa rasio waktu per epoch yang Anda ukur, dan faktor apa yang diduga membuat keduanya berbeda? TODO
5. Receptive field keluaran blok kedua adalah $10\times10$ piksel. Apakah ukuran ini cukup untuk mengenali objek berukuran $32\times32$, dan layer mana yang menggabungkan informasi di luar wilayah tersebut? TODO

## Checklist sebelum mengumpulkan

- [ ] Identitas, seed, versi library, device, dan dataset tercantum.
- [ ] Setiap hitungan tangan berada pada sel sebelum kode verifikasinya.
- [ ] Implementasi manual tidak memanggil `F.conv2d` maupun autograd.
- [ ] Seluruh sel `cek` dan `assert` lolos pada `float64`.
- [ ] Tabel anatomi memuat shape, parameter, perkalian, dan receptive field untuk kedua model.
- [ ] Kedua run memakai data, seed, dan 790 update yang sama.
- [ ] Test set hanya dipanggil satu kali pada Tugas T4.
- [ ] `metrics.csv` memuat kedua run.
- [ ] Notebook lolos *Restart Kernel and Run All*.